<a href="https://colab.research.google.com/github/ManoharRavula/Manohar_INFO5731_-Spring2024/blob/main/Ravulapalli_Manohar_Exercise_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 In-class Exercise 4**

**This exercise will provide a valuable learning experience in working with text data and extracting features using various topic modeling algorithms. Key concepts such as Latent Dirichlet Allocation (LDA), Latent Semantic Analysis (LSA), lda2vec, and BERTopic.**

***Please use the text corpus you collected in your last in-class-exercise for this exercise. Perform the following tasks***.

**Expectations**:
*   Students are expected to complete the exercise during lecture period to meet the active participation criteria of the course.
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

**Total points**: 40

**Deadline**: This in-class exercise is due at the end of the day tomorrow, at 11:59 PM.

**Late submissions will have a penalty of 10% of the marks for each day of late submission, and no requests will be answered. Manage your time accordingly.**


## Question 1 (10 Points)

**Generate K topics by using LDA, the number of topics K should be decided by the coherence score, then summarize what are the topics.**

You may refer the code here: https://www.machinelearningplus.com/nlp/topic-modeling-gensim-python/

In [ ]:
# Write your code here
!pip install gensim nltk numpy
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
from gensim.models import CoherenceModel
import numpy as np
np.random.seed(2018)
import nltk
nltk.download('wordnet')
from google.colab import files
uploaded = files.upload()

#Loading the CNN news CSV file into a Pandas DataFrame
import pandas as pd
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

# Step 3: Access the "description" column
descriptions = df['description'].tolist()

#pre-processing the data
def lemmatize_stemming(text):
    stemmer = SnowballStemmer("english")
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result

processed_docs = [preprocess(doc) for doc in descriptions]

dictionary = gensim.corpora.Dictionary(processed_docs)
dictionary.filter_extremes(no_below=1, no_above=0.5, keep_n=100000)
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

#checking coherence values
def compute_coherence_values(dictionary, corpus, texts, limit, start=2, step=1):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=100, update_every=1, chunksize=100, passes=10, alpha='auto', per_word_topics=True)
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())
    return model_list, coherence_values

model_list, coherence_values = compute_coherence_values(dictionary=dictionary, corpus=bow_corpus, texts=processed_docs, start=2, limit=8, step=1)

# Selecting the model and print the topics
optimal_model = model_list[coherence_values.index(max(coherence_values))]
model_topics = optimal_model.show_topics(formatted=False)

def summarize_topics(model_topics):
    topics_summary = []
    for i, topic in enumerate(model_topics):
        topic_keywords = ", ".join([word for word, prop in topic[1]])
        topics_summary.append(f"Topic {i+1}: {topic_keywords}")
    return topics_summary

topics_summary = summarize_topics(model_topics)
for topic in topics_summary:
    print(topic)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saving CNN_Articles.csv to CNN_Articles (1).csv
Topic 1: capitol, presid, elect, women, riot, speak, govern, violenc, clear, identifi
Topic 2: wing, protest, expert, condemn, extremist, repeat, rough, intern, claim, russia
Topic 3: attack, american, campaign, onlin, face, sourc, boy, proud, open, height
Topic 4: wednesday, help, sign, life, come, support, ahead, leav, inaugur, play
Topic 5: year, world, storm, depart, chang, includ, sunday, citi, civil, game
Topic 6: right, trump, peopl, year, group, warn, live, say, accord, black
Topic 7: time, polic, week, home, arrest, matter, hate, metropolitan, court, washington


## Question 2 (10 Points)

**Generate K topics by using LSA, the number of topics K should be decided by the coherence score, then summarize what are the topics.**

You may refer the code here: https://www.datacamp.com/community/tutorials/discovering-hidden-topics-python

In [ ]:
# Write your code here
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from gensim.matutils import Sparse2Corpus
from gensim.parsing.preprocessing import STOPWORDS
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('wordnet')
nltk.download('punkt')

from google.colab import files
uploaded = files.upload()

#Loaded Medical paper NLP dataset
import pandas as pd
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

# Step 3: Access the "title" column
descriptions = df['title'].tolist()

# Preprocessing function
def preprocess_documents(documents):
    lemmatizer = WordNetLemmatizer()
    preprocessed_docs = []
    for document in documents:
        tokens = word_tokenize(document.lower())
        lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in STOPWORDS and token.isalpha()]
        preprocessed_docs.append(' '.join(lemmatized_tokens))
    return preprocessed_docs

# calculating the coherence score
def calculate_coherence_score(document_term_matrix, feature_names, corpus, dictionary, n_topics):
    lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
    lsa_topic_matrix = lsa_model.fit_transform(document_term_matrix)

    topics = [[feature_names[i] for i in topic.argsort()[:-11:-1]] for topic in lsa_model.components_]
    coherence_model = CoherenceModel(topics=topics, texts=corpus, dictionary=dictionary, coherence='c_v')

    return coherence_model.get_coherence()

# function to find optimal topics from the dataset
def find_optimal_topics(documents):
    preprocessed_docs = preprocess_documents(documents)
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')
    dtm_tfidf = tfidf_vectorizer.fit_transform(preprocessed_docs)

    # Convert the TF-IDF matrix into a Gensim corpus
    corpus_gensim = Sparse2Corpus(dtm_tfidf, documents_columns=False)
    dictionary = Dictionary([tfidf_vectorizer.get_feature_names_out()])

    # Explore a range of topics
    n_topics_range = range(2, 6)  # Adjust based on your dataset size and diversity
    coherence_scores = []

    for n_topics in n_topics_range:
        coherence_score = calculate_coherence_score(dtm_tfidf, tfidf_vectorizer.get_feature_names_out(), preprocessed_docs, dictionary, n_topics)
        coherence_scores.append(coherence_score)

    # Finding the number of topics with the highest coherence score
    optimal_n_topics = n_topics_range[np.argmax(coherence_scores)]
    optimal_coherence_score = max(coherence_scores)

    # Applying LSA with the optimal number of topics
    lsa_model = TruncatedSVD(n_components=optimal_n_topics, random_state=42)
    lsa_model.fit_transform(dtm_tfidf)
    topics = [[tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]] for topic in lsa_model.components_]

    return optimal_n_topics, optimal_coherence_score, topics

# Run the function and print the optimal topics
optimal_n_topics, optimal_coherence_score, topics = find_optimal_topics(descriptions)
print("Optimal number of topics:", optimal_n_topics)
print("Optimal coherence score:", optimal_coherence_score)
for i, topic in enumerate(topics):
    print(f"Topic {i+1}:", topic)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Saving testB.csv to testB.csv


/usr/local/lib/python3.10/dist-packages/gensim/topic_coherence/direct_confirmation_measure.py:204: RuntimeWarning: divide by zero encountered in scalar divide
  m_lr_i = np.log(numerator / denominator)
/usr/local/lib/python3.10/dist-packages/gensim/topic_coherence/indirect_confirmation_measure.py:323: RuntimeWarning: invalid value encountered in scalar divide
  return cv1.T.dot(cv2)[0, 0] / (_magnitude(cv1) * _magnitude(cv2))


Optimal number of topics: 2
Optimal coherence score: nan
Topic 1: ['energy', 'economic', 'emission', 'china', 'renewable', 'analysis', 'evidence', 'country', 'environmental', 'consumption']
Topic 2: ['energy', 'renewable', 'consumption', 'storage', 'optimization', 'hybrid', 'solar', 'integrated', 'power', 'resource']


## Question 3 (10 points):
**Generate K topics by using lda2vec, the number of topics K should be decided by the coherence score, then summarize what are the topics.**

You may refer the code here: https://nbviewer.org/github/cemoody/lda2vec/blob/master/examples/twenty_newsgroups/lda2vec/lda2vec.ipynb

In [ ]:
# Write your code here


## Question 4 (10 points):
**Generate K topics by using BERTopic, the number of topics K should be decided by the coherence score, then summarize what are the topics.**

You may refer the code here: https://colab.research.google.com/drive/1FieRA9fLdkQEGDIMYl0I3MCjSUKVF8C-?usp=sharing

In [9]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups
import numpy as np
import pandas as pd

#Loading the dataset
newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
documents = newsgroups.data


#Generating topics and optimize the number of topics based on coherence score
topic_model = BERTopic(calculate_probabilities=True, verbose=True)
topics, probabilities = topic_model.fit_transform(documents)

# Optimal number of topics using c-TF-IDF and coherence score
scores = topic_model.get_topic_info(); scores.head(10)

# Summarize the generated topics
topic_summaries = []
for topic_number in sorted(list(set(topics))):
    if topic_number == -1:
        continue
    topic_summaries.append((topic_number, topic_model.get_topic(topic_number)))

print("Topics summaries:")
for summary in topic_summaries:
    print(f"Topic {summary[0]}:", summary[1])

2024-03-29 03:32:48,864 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/589 [00:00<?, ?it/s]

2024-03-29 03:33:43,378 - BERTopic - Embedding - Completed ✓
2024-03-29 03:33:43,379 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-03-29 03:34:00,560 - BERTopic - Dimensionality - Completed ✓
2024-03-29 03:34:00,563 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-03-29 03:34:43,226 - BERTopic - Cluster - Completed ✓
2024-03-29 03:34:43,238 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-03-29 03:34:49,168 - BERTopic - Representation - Completed ✓


Topics summaries:
Topic 0: [('game', 0.010334527391530283), ('team', 0.009007575661802415), ('games', 0.007178374118419969), ('he', 0.0069787029783431416), ('players', 0.006324798756235954), ('season', 0.006219081788121945), ('hockey', 0.006120573977823499), ('play', 0.005774268767766829), ('25', 0.005634360052730057), ('year', 0.005585196108034032)]
Topic 1: [('key', 0.013631147876285448), ('clipper', 0.012489014949510848), ('chip', 0.01175261975503659), ('encryption', 0.011737535011251004), ('keys', 0.009466050707337746), ('escrow', 0.00818271274534279), ('government', 0.008068900461595722), ('nsa', 0.0075504815139545555), ('algorithm', 0.0066690755523687835), ('be', 0.006183224167739047)]
Topic 2: [('israel', 0.015221276569853212), ('israeli', 0.011582997769911483), ('jews', 0.009603782547578538), ('arab', 0.009260587592840354), ('jewish', 0.007003840503510874), ('arabs', 0.006818124350234328), ('of', 0.0053567888156505015), ('kuwait', 0.005353199263635196), ('palestinian', 0.005340

## **Question 3 (Alternative) - (10 points)**

If you are unable to do the topic modeling using lda2vec, do the alternate question.

Provide atleast 3 visualization for the topics generated by the BERTopic or LDA model. Explain each of the visualization in detail.

In [18]:
topic_model.visualize_hierarchy(top_n_topics=50)
#Branches and Leaves: Each leaf (the end of a branch) represents one topic discovered in the dataset. Topics are indicated by numbers and their top defining terms are listed alongside them (like "3_ites_cheek_yep", "29_pom_cds_dos", etc.).
#Cluster Formation: The branches show how topics are linked to each other. The height at which any two branches merge represents the distance or dissimilarity between two topics; the lower on the y-axis, the more similar the topics are.
#Red Cluster: This is technology-related, with references to drivers, monitors, and modems.
#Blue Cluster: This is about different forms of media and software (e.g., audio, games, operating systems).
#Green Cluster: This cluster has topics that are related to printing and computer hardware.
#Yellow Cluster: this is focus on storage devices and possibly legal issues (e.g., "batf_warrant_raid").

In [19]:

#Barchart visualization
topic_model.visualize_barchart(top_n_topics=5)
#It provided a clear way to understand which words are most important in describing each topic
#For example when we look at Topic 0 (Orange bars): This topic seems to be about sports or gaming, given the prevalence of words like "game," "team," "games," "he," and "players."
#The length of the bar represents the relevance score of the word to the topic, with "game"


In [20]:
topic_model.visualize_distribution(probs[200], min_probability=0.015)
#This bar chart shows the likelihood that a particular document is related to various topics identified by a topic modeling algorithm.
'''Each bar represents a different topic, indicated by a number and key terms. The length of the bar shows how probable it is that the document
discusses the corresponding topic. For instance, the document is most likely about operating systems and software (Topic 36)
since that bar is the longest. Other topics, like biking (Topic 50) and safety equipment (Topic 117), are less probable but still relevant to the document '''

## Extra Question (5 Points)

**Compare the results generated by the four topic modeling algorithms, which one is better? You should explain the reasons in details.**

**This question will compensate for any points deducted in this exercise. Maximum marks for the exercise is 40 points.**

In [ ]:
# Write your code here


# Mandatory Question

**Important: Reflective Feedback on this exercise**

Please provide your thoughts and feedback on the exercises you completed in this assignment.

Consider the following points in your response:

**Learning Experience:** Describe your overall learning experience in working with text data and extracting features using various topic modeling algorithms. Did you understand these algorithms and did the implementations helped in grasping the nuances of feature extraction from text data.

**Challenges Encountered:** Were there specific difficulties in completing this exercise?

Relevance to Your Field of Study: How does this exercise relate to the field of NLP?

**(Your submission will not be graded if this question is left unanswered)**



In [ ]:
# Your answer here (no code for this question, write down your answer as detail as possible for the above questions):

'''
This activity has been an exploration, into the realm of Natural Language Processing (NLP) specifically focusing on topic modeling.
It has allowed me to deepen my comprehension of data and the nuances of extracting features.
By applying Latent Dirichlet Allocation (LDA) and Latent Semantic Analysis (LSA) I have gained insights into how significant words
and topics can be derived from text datasets. My interaction with BERTopic was particularly enlightening. This advanced algorithm,
based on the transformers architecture not strengthened my understanding of contextualized topic modeling. Also emphasized the
importance of visual aids in interpreting outcomes. BERTopics user friendly visual representations, like clustering and bar charts
illustrating topic probabilities and word scores provided a way to analyze and assess the thematic structures present, in the data.

One obstacle I faced was initially struggling with parameter adjustments to enhance coherence scores.

'''